# UNIVERSIDAD AUTÓNOMA DEL ESTADO DE MÉXICO

**Asignatura:** Investigación de Operaciones

**Práctica:** Ejercicio 2: Ruta más corta del nodo 1 al nodo 7 mediante el método de la Gran M

**Nombre:** SERGIO MANUEL MARTÍNEZ CORTÉS

**Tema:** Problema de ruta más corta como modelo de programación lineal.

## Introducción

En este ejercicio se busca determinar el recorrido de menor costo entre un nodo de origen y uno de destino. Cada arco dirigido representa una conexión y su número asociado es la distancia o costo.

Se formula el problema como programación lineal. Para obtener una base inicial se agregan variables artificiales penalizadas con **M**, un número positivo muy grande. El algoritmo trabaja simbólicamente con el coeficiente de M y la parte constante.

## 1. Datos de la red

La tabla contiene los arcos dirigidos y sus costos. Las conexiones solo se pueden recorrer en el sentido indicado.

In [7]:
import numpy as np
import pandas as pd
from IPython.display import display

arcos = [(1, 2, 4), (1, 3, 3), (2, 3, 2), (2, 4, 5), (2, 5, 6), (3, 4, 3), (3, 5, 4), (4, 5, 1), (4, 6, 5), (5, 6, 2), (5, 7, 7), (6, 7, 3)]
origen, destino = 1, 7
nodos = sorted({v for i, j, _ in arcos for v in (i, j)}, key=lambda x: str(x))
nombres = [f"x{i}{j}" for i, j, _ in arcos]
c = np.array([costo for _, _, costo in arcos], dtype=float)

display(pd.DataFrame(arcos, columns=["Nodo inicial", "Nodo final", "Costo"]))
print("Origen:", origen, "| Destino:", destino)
print("Nodos:", nodos)
print("Número de arcos:", len(arcos))

,Nodo inicial,Nodo final,Costo
0,1,2,4
1,1,3,3
2,2,3,2
3,2,4,5
4,2,5,6
5,3,4,3
6,3,5,4
7,4,5,1
8,4,6,5
9,5,6,2


Origen: 1 | Destino: 7
Nodos: [1, 2, 3, 4, 5, 6, 7]
Número de arcos: 12


## 2. Formulación del modelo

- **Función objetivo:** minimizar la suma de los costos de los arcos seleccionados.
- **Origen:** flujo que sale menos flujo que entra = 1.
- **Nodos intermedios:** flujo que entra menos flujo que sale = 0.
- **Destino:** flujo que entra menos flujo que sale = 1.
- **Variables:** xᵢⱼ ≥ 0.

La ecuación del destino se omite porque es dependiente de las demás ecuaciones de conservación de flujo.

In [8]:
# Matriz de restricciones (se omite la ecuación redundante del destino)
filas = []
lado_derecho = []
for nodo in nodos:
    if nodo == destino:
        continue
    fila = []
    for i, j, costo in arcos:
        if nodo == origen:
            coef = (1 if i == nodo else 0) - (1 if j == nodo else 0)
            rhs = 1
        else:
            coef = (1 if j == nodo else 0) - (1 if i == nodo else 0)
            rhs = 0
        fila.append(coef)
    filas.append(fila)
    lado_derecho.append(rhs)

A = np.array(filas, dtype=float)
b = np.array(lado_derecho, dtype=float)

print("Función objetivo:")
print("Minimizar Z =", " + ".join(f"{int(v)}{name}" for name, v in zip(nombres, c)))
print("\nRestricciones:")
for k, nodo in enumerate([n for n in nodos if n != destino]):
    terminos = [f"{'+' if v > 0 else '-'} {nombres[j]}" for j, v in enumerate(A[k]) if v != 0]
    print(f"Nodo {nodo}: {' '.join(terminos)} = {int(b[k])}")
print("\nDimensión de A:", A.shape)

Función objetivo:
Minimizar Z = 4x12 + 3x13 + 2x23 + 5x24 + 6x25 + 3x34 + 4x35 + 1x45 + 5x46 + 2x56 + 7x57 + 3x67

Restricciones:
Nodo 1: + x12 + x13 = 1
Nodo 2: + x12 - x23 - x24 - x25 = 0
Nodo 3: + x13 + x23 - x34 - x35 = 0
Nodo 4: + x24 + x34 - x45 - x46 = 0
Nodo 5: + x25 + x35 + x45 - x56 - x57 = 0
Nodo 6: + x46 + x56 - x67 = 0

Dimensión de A: (6, 12)


## 3. Método de la Gran M

Se agrega una variable artificial por cada restricción. La función objetivo ampliada queda:

**Min Z = Σ(cᵢⱼ xᵢⱼ) + M(a₁ + a₂ + … + aₘ)**

Las variables artificiales forman la base inicial. Durante las iteraciones se busca que salgan de la base, ya que tienen una penalización muy alta.

In [9]:
def simplex_gran_m(c, A, b, tol=1e-9, max_iter=300):
    m, n = A.shape
    Af = np.hstack([A, np.eye(m)])
    cM = np.r_[np.zeros(n), np.ones(m)]
    c0 = np.r_[c, np.zeros(m)]
    base = list(range(n, n + m))
    pasos = []
    for _ in range(max_iter):
        Binv = np.linalg.inv(Af[:, base])
        T = Binv @ Af
        xB = Binv @ b
        rM = cM - cM[base] @ T
        r0 = c0 - c0[base] @ T
        zM = float(cM[base] @ xB)
        z0 = float(c0[base] @ xB)
        no_basicas = [j for j in range(n) if j not in base]
        paso = {"base": base.copy(), "xB": xB.copy(), "rM": rM.copy(),
                "r0": r0.copy(), "zM": zM, "z0": z0,
                "entra": None, "sale": None}
        negativas = [j for j in no_basicas
                     if rM[j] < -tol or (abs(rM[j]) <= tol and r0[j] < -tol)]
        if not negativas:
            pasos.append(paso)
            return pasos
        entra = min(negativas, key=lambda j: (round(rM[j], 9), round(r0[j], 9), j))
        razones = [(xB[i] / T[i, entra], i) for i in range(m) if T[i, entra] > tol]
        if not razones:
            raise ValueError("El problema no está acotado.")
        minimo = min(v for v, _ in razones)
        fila = min(i for v, i in razones if abs(v - minimo) < tol)
        sale = base[fila]
        paso.update({"entra": entra, "sale": sale, "fila": fila})
        pasos.append(paso)
        base[fila] = entra
    raise RuntimeError("Se alcanzó el máximo de iteraciones.")

## 4. Iteraciones del método

In [10]:
pasos = simplex_gran_m(c, A, b)
print("Total de iteraciones:", len(pasos) - 1)
for k, p in enumerate(pasos):
    if p["entra"] is None:
        print(f"Iteración {k}: solución óptima alcanzada.")
    else:
        nombre_entra = nombres[p["entra"]]
        nombre_sale = (f"a{p['sale'] - len(arcos) + 1}"
                       if p["sale"] >= len(arcos) else nombres[p["sale"]])
        print(f"Iteración {k}: entra {nombre_entra}, sale {nombre_sale}.")

Total de iteraciones: 6
Iteración 0: entra x13, sale a3.
Iteración 1: entra x12, sale a2.
Iteración 2: entra x34, sale a4.
Iteración 3: entra x35, sale a5.
Iteración 4: entra x56, sale a6.
Iteración 5: entra x67, sale a1.
Iteración 6: solución óptima alcanzada.


## 5. Solución óptima

In [11]:
final = pasos[-1]
solucion = np.zeros(len(arcos))
for indice, valor in zip(final["base"], final["xB"]):
    if indice < len(arcos):
        solucion[indice] = valor

seleccionados = [(i, j, costo) for (i, j, costo), valor in zip(arcos, solucion)
                 if valor > 0.5]
print("Arcos seleccionados:")
for i, j, costo in seleccionados:
    print(f"{i} -> {j} | costo: {costo}")
print("\nCosto mínimo Z =", round(final["z0"], 6))
print("Variables artificiales en la base:",
      [f"a{idx-len(arcos)+1}" for idx in final["base"] if idx >= len(arcos)])

Arcos seleccionados:
1 -> 3 | costo: 3
3 -> 5 | costo: 4
5 -> 6 | costo: 2
6 -> 7 | costo: 3

Costo mínimo Z = 12.0
Variables artificiales en la base: []


## 6. Verificación con Dijkstra

Se compara el costo obtenido por Gran M con el algoritmo de Dijkstra, que calcula rutas mínimas en redes con costos no negativos.

In [12]:
# Implementación sencilla de Dijkstra, sin bibliotecas adicionales
import heapq

def dijkstra(arcos, origen, destino):
    ady = {}
    for i, j, costo in arcos:
        ady.setdefault(i, []).append((j, costo))
    dist = {origen: 0}
    anterior = {}
    cola = [(0, origen)]
    while cola:
        costo_actual, nodo = heapq.heappop(cola)
        if costo_actual != dist.get(nodo):
            continue
        if nodo == destino:
            break
        for vecino, peso in ady.get(nodo, []):
            nuevo = costo_actual + peso
            if nuevo < dist.get(vecino, float("inf")):
                dist[vecino] = nuevo
                anterior[vecino] = nodo
                heapq.heappush(cola, (nuevo, vecino))
    if destino not in dist:
        return [], float("inf")
    ruta = [destino]
    while ruta[-1] != origen:
        ruta.append(anterior[ruta[-1]])
    ruta.reverse()
    return ruta, dist[destino]

ruta_dijkstra, costo_dijkstra = dijkstra(arcos, origen, destino)
print("Ruta por Dijkstra:", " -> ".join(map(str, ruta_dijkstra)))
print("Costo por Dijkstra:", costo_dijkstra)
print("¿Coinciden los costos?:", abs(costo_dijkstra - final["z0"]) < 1e-6)

Ruta por Dijkstra: 1 -> 3 -> 5 -> 6 -> 7
Costo por Dijkstra: 12
¿Coinciden los costos?: True


## 7. Conclusión

En esta práctica se representó una red como un modelo de programación lineal y se aplicó el método de la Gran M. Las variables artificiales permitieron iniciar el procedimiento y las iteraciones condujeron a una solución óptima. La comparación con Dijkstra sirve para comprobar el costo mínimo obtenido.